## 4 - Optimization of model hyperparameters 

In [2]:
import sys
import os
from pathlib import Path
import pandas as pd

_parent = Path().resolve().parent
if str(_parent) not in sys.path:
    sys.path.insert(0, str(_parent))

from src.evaluation import evaluate_river

import yaml
cfg = yaml.safe_load(Path('../config/config.yaml').read_text())

path_data     = cfg['path_data']
path_fold     = cfg['path_fold']

splits_info = pd.read_csv(path_fold)
df_raw      = pd.read_csv(path_data)

results_river = evaluate_river(splits_info, df_raw)
print(results_river[['MAE','RMSE','MAPE']].mean())

MAE      4.995843
RMSE     5.871808
MAPE    26.150396
dtype: float64


In [3]:
# grid search simple
for opt in ["sgd", "adam"]:
    for lr in [0.001, 0.01]:  #0.05 explose
        print('lr :', lr, ', opt :', opt)
        results = evaluate_river(splits_info, 
                                    df_raw, 
                                    model_kwargs={"lr": lr,
                                                  "optimizer": opt})
        print(results[['MAE','RMSE', 'R2', 'MAPE']].mean())

lr : 0.001 , opt : sgd
MAE      2.748385
RMSE     3.421358
R2       0.010251
MAPE    13.440065
dtype: float64
lr : 0.01 , opt : sgd
MAE      4.995843
RMSE     5.871808
R2      -2.398060
MAPE    26.150396
dtype: float64
lr : 0.001 , opt : adam
MAE      3.189094
RMSE     3.984529
R2       0.373030
MAPE    14.845934
dtype: float64
lr : 0.01 , opt : adam
MAE      2.755521
RMSE     3.408621
R2      -0.082733
MAPE    13.372295
dtype: float64


In [4]:
best = {"mae": float("inf")}

for l2 in [0.0001, 0.001, 0.01]:
    res = evaluate_river(splits_info, 
                         df_raw,
                         model_kwargs={"lr": 0.01, 
                                       "optimizer": "adam", 
                                       "l2": l2})
    mae = res["MAE"].mean()
    print(f"l2={l2:.4f}  MAE={mae:.4f}")
    if mae < best["mae"]:
        best = {"mae": mae, "l2": l2}

print(f"\nMeilleur : l2={best['l2']}  MAE={best['mae']:.4f}")

l2=0.0001  MAE=2.7555
l2=0.0010  MAE=2.7557
l2=0.0100  MAE=2.7576

Meilleur : l2=0.0001  MAE=2.7555
